# 🐄 Detección de Bovinos con YOLOv8
## Proyecto Universitario — Universidad Nacional Rosario Castellanos
### Ciencias de Datos para Negocios

**Objetivo:** Entrenar una Red Neuronal Convolucional (CNN) basada en YOLOv8 
para detectar y localizar bovinos en imágenes mediante bounding boxes.

**Pipeline:**
1. Verificación del dataset
2. Split Train / Val / Test
3. Configuración del entorno YOLOv8
4. Entrenamiento del modelo
5. Evaluación de métricas
6. Inferencia con imágenes nuevas

**Autor:** Brandon Uriel Garcia Sanchez  
**Institución:** Universidad Nacional Rosario Castellanos  
**Librería principal:** YOLOv8 (Ultralytics) + PyTorch

## 1️⃣ Instalación de Librerías

Instalamos las librerías necesarias para el proyecto:

- **ultralytics**: contiene YOLOv8, el modelo que vamos a entrenar
- **opencv-python**: para manipular y visualizar imágenes
- **matplotlib**: para graficar métricas y resultados
- **PyYAML**: para crear y leer el archivo data.yaml
- **shutil / os / random**: librerías nativas de Python para mover archivos y organizar carpetas

In [ ]:
"""
Dependencias instaladas CV, Yolov8
"""

# Instalamos YOLOv8 y librerías necesarias
# !pip install ultralytics
# !pip install opencv-python
# !pip install matplotlib
# !pip install pyyaml
#!pip install ultralytics


In [1]:
   import sys
   print(sys.executable)
   

C:\Users\Uriel\anaconda3\envs\bovinos\python.exe


In [4]:
# Librerias

import os
import shutil
import random
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

## 2️⃣ Verificación del Dataset

Antes de entrenar verificamos que:
- El número de imágenes y etiquetas coincida
- No haya archivos corruptos
- Las rutas estén correctas

Esto es crítico porque si hay imágenes sin su .txt correspondiente 
o viceversa, el entrenamiento puede fallar o dar resultados incorrectos.

In [2]:

# Rutas donde están mis imágenes
IMAGES_PATH = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataraw"
# Rutas donde están mis etiquetas
LABELS_PATH = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\labels"

# Listo todos los archivos de cada carpeta
imagenes = [f for f in os.listdir(IMAGES_PATH) 
            if f.endswith(('.jpg', '.jpeg', '.png'))]
etiquetas = [f for f in os.listdir(LABELS_PATH) 
             if f.endswith('.txt')]

# Imprimimos el total de imagenes
print(f"Total imágenes:  {len(imagenes)}")
print(f"Total etiquetas: {len(etiquetas)}") # imprimimos el total de etiquetas

# Busco imágenes que no tienen su etiqueta correspondiente
# Estas son mis imágenes de fondo — no tienen bovinos, no necesitan etiqueta
sin_etiqueta = []
for img in imagenes:
    nombre = os.path.splitext(img)[0]
    if nombre + '.txt' not in etiquetas:
        sin_etiqueta.append(img)

print(f"\nImágenes sin etiqueta: {len(sin_etiqueta)}")

Total imágenes:  391
Total etiquetas: 332

Imágenes sin etiqueta: 59


## 3️⃣ Split Train / Val / Test

No puedo entrenar y evaluar el modelo con las mismas imágenes.
Si lo hiciera, el modelo simplemente memorizaría los datos y 
no aprendería a identificar imágenes nuevas.

Por eso divido el dataset en 3 partes:
- **Train 70%** → con estas imágenes el modelo aprende
- **Val 20%** → durante el entrenamiento mide si está mejorando
- **Test 10%** → evaluación final con imágenes que nunca vio

In [6]:

# Aquí le digo a Python dónde están mis archivos
# IMAGES_PATH → mis fotos de bovinos
# LABELS_PATH → los .txt con las anotaciones que generé en Makesense
# DATASET_PATH → aquí se va a crear la estructura final para YOLOv8

IMAGES_PATH = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataraw"
LABELS_PATH = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\labels"
DATASET_PATH = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset"

# YOLOv8 espera una estructura muy específica de carpetas
# Si no existe la creo automáticamente con makedirs
# exist_ok=True significa que si ya existe no manda error

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_PATH, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(DATASET_PATH, split, 'labels'), exist_ok=True)

# Listo solo los archivos de imagen, ignorando cualquier otro tipo de archivo
imagenes = [f for f in os.listdir(IMAGES_PATH) 
            if f.endswith(('.jpg', '.jpeg', '.png'))]

# Mezclo las imágenes de forma aleatoria antes de dividirlas
# Esto evita que el modelo aprenda en un orden específico
# seed(42) es una semilla fija — si alguien más corre este código
# obtendrá exactamente el mismo split que yo
random.seed(42)
random.shuffle(imagenes)

# Calculo cuántas imágenes van en cada parte
# 70% para entrenar, 20% para validar, 10% para probar

total     = len(imagenes)
train_end = int(total * 0.70) # hasta aquí es train
val_end   = int(total * 0.90) # de aquí a aquí es val, el resto es test


# Esta función copia cada imagen y su etiqueta correspondiente
# a la carpeta del split que le toca (train, val o test)
train_imgs = imagenes[:train_end]
val_imgs   = imagenes[train_end:val_end]
test_imgs  = imagenes[val_end:]

# Copio imágenes y sus etiquetas a cada carpeta
def copiar(lista, split):
    for img in lista:
        # Obtengo el nombre sin extensión para buscar su .txt
        nombre = os.path.splitext(img)[0]

        # Copio la imagen a su carpeta de destino
        shutil.copy(
            os.path.join(IMAGES_PATH, img),
            os.path.join(DATASET_PATH, split, 'images', img)
        )
        
        # Busco si existe la etiqueta correspondiente
        # Si no existe es una imagen de fondo y no necesita .txt
        label = nombre + '.txt'
        if label in os.listdir(LABELS_PATH):
            shutil.copy(
                os.path.join(LABELS_PATH, label),
                os.path.join(DATASET_PATH, split, 'labels', label)
            )
# Ejecuto la función para cada split
copiar(train_imgs, 'train')
copiar(val_imgs,   'val')
copiar(test_imgs,  'test')

# Reporte final para confirmar
print(f"Train: {len(train_imgs)} imágene")
print(f"Val:   {len(val_imgs)} imágenes")
print(f"Test:  {len(test_imgs)} imágenes")
print(f"Total: {total} imágenes")

Train: 273 imágenes
Val:   78 imágenes
Test:  40 imágenes
Total: 391 imágenes


## 4️⃣ Creación del data.yaml

YOLOv8 necesita un archivo de configuración llamado data.yaml.
Este archivo le dice al modelo tres cosas fundamentales:
- Dónde están las imágenes de train, val y test
- Cuántas clases tiene el dataset
- Cómo se llama cada clase

Sin este archivo YOLOv8 no sabe qué está buscando ni dónde están los datos.

In [8]:
import yaml

# Rutas absolutas que YOLOv8 necesita para encontrar los datos
data = {
    'train': r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\train\images",
    'val':   r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\val\images",
    'test':  r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\test\images",

    # nc = number of classes → tengo 1 sola clase
    'nc': 1,

    # El nombre de mi clase en el mismo orden que usé al etiquetar
    # El índice 0 corresponde a 'bovino' en los archivos .txt
    'names': ['bovino']
}

# Guardo el archivo en la carpeta del dataset
ruta_yaml = r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\data.yaml"

with open(ruta_yaml, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print("data.yaml creado correctamente")
print(f"Ruta: {ruta_yaml}")

data.yaml creado correctamente
Ruta: C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\data.yaml


## 5️⃣ Entrenamiento del Modelo

Aquí empieza el entrenamiento. Uso YOLOv8 con Transfer Learning,
lo que significa que no entreno desde cero sino que parto de un modelo 
que ya aprendió a detectar objetos generales y lo especializo en bovinos.

El modelo base que uso es yolov8n.pt (nano) — el más ligero de YOLOv8.
Es ideal para un dataset de este tamaño y para correr en una CPU sin GPU dedicada.

In [3]:
# Cargo el modelo base preentrenado en COCO
# yolov8n.pt = YOLOv8 nano, el más ligero y rápido
# Si tuvieras GPU podríamos usar yolov8s.pt o yolov8m.pt
model = YOLO('yolov8n.pt')

# Entreno el modelo con mi dataset
model.train(
    data=r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\data.yaml",
    
    epochs=50,       # Número de veces que el modelo ve todo el dataset
    imgsz=640,       # Tamaño al que se redimensionan las imágenes
    batch=8,         # Cuántas imágenes procesa al mismo tiempo
    name='bovinos',  # Nombre del experimento para identificar los resultados
    patience=10      # Si después de 10 epochs no mejora, para automáticamente
)

print("Entrenamiento finalizado")

Ultralytics 8.4.32  Python-3.11.15 torch-2.11.0+cpu CPU (AMD Ryzen 5 5600G with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Uriel\Documents\Bovinos Red Neuronal\dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bovinos, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

## Resultados del Entrenamiento

Visualización de las métricas obtenidas durante las 50 epochs de entrenamiento.

In [5]:
# Métricas reales de tu entrenamiento epoch por epoch
epochs = list(range(1, 51))

mAP50 = [0.610, 0.423, 0.308, 0.591, 0.565, 0.576, 0.690, 0.567, 0.706, 0.680,
         0.754, 0.779, 0.746, 0.697, 0.688, 0.738, 0.773, 0.779, 0.775, 0.811,
         0.803, 0.809, 0.810, 0.822, 0.827, 0.780, 0.787, 0.809, 0.831, 0.849,
         0.835, 0.853, 0.839, 0.841, 0.846, 0.843, 0.866, 0.851, 0.840, 0.824,
         0.821, 0.824, 0.843, 0.847, 0.836, 0.820, 0.827, 0.843, 0.845, 0.850]

precision = [0.878, 0.541, 0.409, 0.699, 0.690, 0.545, 0.656, 0.684, 0.622, 0.710,
             0.758, 0.790, 0.731, 0.650, 0.637, 0.591, 0.756, 0.748, 0.761, 0.729,
             0.703, 0.765, 0.803, 0.837, 0.806, 0.769, 0.756, 0.728, 0.787, 0.807,
             0.775, 0.855, 0.778, 0.847, 0.842, 0.822, 0.849, 0.835, 0.817, 0.821,
             0.829, 0.858, 0.844, 0.868, 0.842, 0.834, 0.847, 0.812, 0.811, 0.818]

recall = [0.143, 0.376, 0.337, 0.507, 0.535, 0.574, 0.693, 0.525, 0.683, 0.653,
          0.703, 0.693, 0.726, 0.752, 0.693, 0.812, 0.772, 0.764, 0.752, 0.798,
          0.812, 0.782, 0.752, 0.733, 0.762, 0.802, 0.812, 0.832, 0.772, 0.782,
          0.784, 0.762, 0.842, 0.802, 0.832, 0.822, 0.837, 0.851, 0.802, 0.812,
          0.822, 0.776, 0.782, 0.782, 0.782, 0.802, 0.821, 0.832, 0.850, 0.846]

# Configuración visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    ax.title.set_color('white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

# Gráfica 1 — mAP50
axes[0].plot(epochs, mAP50, color='#58a6ff', linewidth=2)
axes[0].fill_between(epochs, mAP50, alpha=0.15, color='#58a6ff')
axes[0].axhline(y=0.85, color='#f85149', linestyle='--', linewidth=1.2, label='Final: 0.850')
axes[0].set_title('mAP50 por Epoch', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('mAP50')
axes[0].set_ylim(0, 1)
axes[0].legend(facecolor='#21262d', labelcolor='white', edgecolor='#30363d')
axes[0].grid(True, color='#21262d', linewidth=0.5)

# Gráfica 2 — Precision y Recall
axes[1].plot(epochs, precision, color='#3fb950', linewidth=2, label='Precision')
axes[1].plot(epochs, recall, color='#d29922', linewidth=2, label='Recall')
axes[1].set_title('Precision & Recall por Epoch', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1)
axes[1].legend(facecolor='#21262d', labelcolor='white', edgecolor='#30363d')
axes[1].grid(True, color='#21262d', linewidth=0.5)

# Título general
fig.suptitle('Detección de Bovinos — YOLOv8n | 50 Epochs | 391 imágenes',
             fontsize=14, fontweight='bold', color='white', y=1.02)

plt.tight_layout()
plt.savefig(r"C:\Users\Uriel\Documents\Bovinos Red Neuronal\resultados_entrenamiento.png",
            dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("Gráfica guardada en tu carpeta del proyecto")

<Figure size 1400x500 with 2 Axes>

Gráfica guardada en tu carpeta del proyecto
